# dataclasses-replace-args — worked example 2: Build a name-tagged sweep from a dict of override dicts

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclasses-replace-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common wandb pattern is a *named* sweep: a mapping from run-name to the overrides that define that run. Each value is itself a dict of field overrides applied with `dataclasses.replace(base, **override_dict)`. The result preserves insertion order of the outer dict (Python 3.7+) and never mutates the shared base.

## Worked solution

**Goal:** turn `{'baseline': {}, 'fast': {'lr': 1e-3}, 'big-batch': {'batch_size': 128}}` into a list of `(name, TrainingArgs)` pairs.

1. **Iterate the outer dict in order.** `dict.items()` yields keys in insertion order, so the returned list is deterministic and matches the config the user wrote.
2. **Each value is an override dict.** For `'baseline'` the override dict is empty, so `replace(base, **{})` returns a faithful clone of base with no changes (but still a new object).
3. **Splat per entry.** `replace(base, **ov)` applies that run's overrides only; all other fields fall back to base. `'fast'` changes only `lr`; `'big-batch'` changes only `batch_size`.
4. **Pair the name with the variant.** We emit `(name, variant)` tuples so downstream code can log each run under its name.
5. **No shared mutation.** Every variant is a distinct object built from the same untouched base, which is exactly what you want before launching parallel sweep runs.

The print loop shows each name alongside its resolved lr and batch_size.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def make_named_sweep(base, named_overrides):
    return [(name, replace(base, **ov)) for name, ov in named_overrides.items()]

base = TrainingArgs()
sweep = make_named_sweep(base, {
    'baseline': {},
    'fast': {'lr': 1e-3},
    'big-batch': {'batch_size': 128},
})
for name, v in sweep:
    print(f'{name}: lr={v.lr} batch_size={v.batch_size}')
print('base still default lr =', base.lr)